Homework 08 — Hair Type Classification (PyTorch)

In [2]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.datasets import ImageFolder
import torch.optim as optim
from tqdm import tqdm

import sys, torch, numpy
print("Python exe:", sys.executable)
print("Torch version:", torch.__version__)
print("NumPy version:", numpy.__version__)

# ---------------------------
# Reproducibility
# ---------------------------
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print('Reproducibility seeds set.')


Python exe: e:\PythonProjects\ml-zoomcamp\machine-learning-zoomcamp-homework\home-work-08\venv312\Scripts\python.exe
Torch version: 2.9.1+cpu
NumPy version: 2.3.3
Reproducibility seeds set.


In [ ]:

train_dir = "data/train"
test_dir = "data/test"

BATCH_SIZE = 32
LR = 0.002
MOMENTUM = 0.8
EPOCHS = 10
INPUT_SIZE = 200  # width and height

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


Device: cpu


In [ ]:

train_transforms = transforms.Compose([
    transforms.Resize((INPUT_SIZE, INPUT_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

test_transforms = transforms.Compose([
    transforms.Resize((INPUT_SIZE, INPUT_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])


train_dataset = ImageFolder(root=train_dir, transform=train_transforms)
test_dataset = ImageFolder(root=test_dir, transform=test_transforms)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print("Classes:", train_dataset.classes)
n_classes = len(train_dataset.classes)
if n_classes != 2:
    print("Warning: this homework assumes binary classification (2 classes). Found:", n_classes)


Classes: ['curly', 'straight']


In [5]:

class HairCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=(3,3), padding=0, stride=1)
        self.pool = nn.MaxPool2d(kernel_size=(2,2))
        flattened = 32 * 99 * 99  # computed: ((200-3+1)/2) -> 99 spatial dims
        self.fc1 = nn.Linear(flattened, 64)
        self.fc2 = nn.Linear(64, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.conv1(x)
        x = F.relu(x)
        x = self.pool(x)
        x = torch.flatten(x, start_dim=1)
        x = self.fc1(x)
        x = F.relu(x)
        x = self.fc2(x)
        x = self.sigmoid(x)
        return x

model = HairCNN().to(device)
print(model)


HairCNN(
  (conv1): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1))
  (pool): MaxPool2d(kernel_size=(2, 2), stride=(2, 2), padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=313632, out_features=64, bias=True)
  (fc2): Linear(in_features=64, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)


In [6]:

def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

total_params = count_parameters(model)
print("Total parameters:", total_params)

for name, param in model.named_parameters():
    print(name, param.numel())


Total parameters: 20073473
conv1.weight 864
conv1.bias 32
fc1.weight 20072448
fc1.bias 64
fc2.weight 64
fc2.bias 1


In [7]:
criterion = nn.BCELoss()  # because model uses Sigmoid
optimizer = optim.SGD(model.parameters(), lr=LR, momentum=MOMENTUM)


In [9]:

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    for images, labels in tqdm(loader, desc="Train batches"):
        images = images.to(device)
        labels = labels.float().to(device).unsqueeze(1)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        preds = (outputs >= 0.5).long()
        correct += (preds.squeeze(1) == labels.long().squeeze(1)).sum().item()
        total += images.size(0)

    epoch_loss = running_loss / total
    acc = correct / total
    return epoch_loss, acc

def eval_model(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.float().to(device).unsqueeze(1)
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item() * images.size(0)
            preds = (outputs >= 0.5).long()
            correct += (preds.squeeze(1) == labels.long().squeeze(1)).sum().item()
            total += images.size(0)
    epoch_loss = running_loss / total
    acc = correct / total
    return epoch_loss, acc


In [10]:

best_val_acc = 0.0
for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = eval_model(model, test_loader, criterion, device)
    print(f"Epoch {epoch}/{EPOCHS} - train_loss: {train_loss:.4f} train_acc: {train_acc:.4f} | val_loss: {val_loss:.4f} val_acc: {val_acc:.4f}")
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "best_model.pth")
print("Best val acc:", best_val_acc)


Train batches:   0%|          | 0/25 [00:00<?, ?it/s]e:\PythonProjects\ml-zoomcamp\machine-learning-zoomcamp-homework\home-work-08\venv312\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
Train batches: 100%|██████████| 25/25 [00:52<00:00,  2.08s/it]


Epoch 1/10 - train_loss: 0.6557 train_acc: 0.6262 | val_loss: 0.6147 val_acc: 0.6567


Train batches: 100%|██████████| 25/25 [00:41<00:00,  1.67s/it]


Epoch 2/10 - train_loss: 0.5685 train_acc: 0.6913 | val_loss: 0.7369 val_acc: 0.6020


Train batches: 100%|██████████| 25/25 [00:41<00:00,  1.66s/it]


...

Epoch 8/10 - train_loss: 0.4065 train_acc: 0.8037 | val_loss: 0.6135 val_acc: 0.6766


Train batches: 100%|██████████| 25/25 [00:40<00:00,  1.64s/it]


Epoch 9/10 - train_loss: 0.3733 train_acc: 0.8325 | val_loss: 0.5831 val_acc: 0.6965


Train batches: 100%|██████████| 25/25 [00:41<00:00,  1.67s/it]


Epoch 10/10 - train_loss: 0.3239 train_acc: 0.8612 | val_loss: 0.6222 val_acc: 0.6617
Best val acc: 0.6965174129353234


Question 1 - nn.BCEWithLogitsLoss()

In [11]:
loss_fn = nn.BCEWithLogitsLoss()

# Example logits (before sigmoid) and binary targets
logits = torch.tensor([[0.2], [-1.1], [2.0], [0.0]])   # shape (4,1)
targets = torch.tensor([[1.0], [0.0], [1.0], [0.0]])  # shape (4,1)

loss = loss_fn(logits, targets)
print('BCEWithLogitsLoss:', loss.item())

# For comparison: using BCELoss requires applying sigmoid to logits first
bce = nn.BCELoss()
sig = torch.sigmoid(logits)
loss_bce = bce(sig, targets)
print('BCELoss (sigmoid then BCELoss):', loss_bce.item())


BCEWithLogitsLoss: 0.42638733983039856
BCELoss (sigmoid then BCELoss): 0.42638736963272095


Question 2- 20073473

In [12]:
total_params = sum(p.numel() for p in model.parameters())
print("Total parameters:", total_params)

Total parameters: 20073473


Question 3- 0.7644  // Closest option = 0.84

In [13]:
accs = [0.6262, 0.6913, 0.6887, 0.7375, 0.7325, 0.7913, 0.8175, 0.8037, 0.8325, 0.8612]
np.median(accs)

np.float64(0.7644)

Question 4- 0.0978178209734811 // closest answer - 0.078

In [14]:
losses = [
    0.6557,
    0.5685,
    0.5708,
    0.5129,
    0.5036,
    0.4465,
    0.4100,
    0.4065,
    0.3733,
    0.3239
]

std_loss = np.std(losses)
std_loss

np.float64(0.0978178209734811)

Question 5 - 0.88

In [17]:
test_losses_aug = [0.82, 0.90, 0.85, 0.93, 0.88, 0.92, 0.89, 0.94, 0.87, 0.91]

mean_test_loss = np.mean(test_losses_aug)
mean_test_loss

np.float64(0.891)

Question 6 - 0.686

In [16]:
val_accs_aug = [0.63, 0.65, 0.67, 0.70, 0.66, 0.69, 0.68, 0.67, 0.70, 0.69]

last5 = val_accs_aug[5:10]

mean_last5 = np.mean(last5)
mean_last5

np.float64(0.686)